In [1]:
!git clone https://github.com/EleutherAI/lm-evaluation-harness
%cd lm-evaluation-harness
!pip install -e .
!pip install accelerate bitsandbytes

fatal: destination path 'lm-evaluation-harness' already exists and is not an empty directory.
/kaggle/working/lm-evaluation-harness
Obtaining file:///kaggle/working/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 7.6 MB/s eta 0:00:00
  Building editable for lm_eval (pyproject.toml) ... done
  Created wheel for lm_eval: filename=lm_eval-0.4.14.dev0-0.editable-py3-none-any.whl size=32334 sha256=0a77fff2a023763cbefe3f595873b5e474ee0066c90c5

In [2]:
# !pip install -q "transformers==4.57.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


# mistral

In [2]:
pip install -U "transformers>=5.0.0" accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 82.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 26.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.29.0:
      Successfully uninstalled huggingface_hub-1.29.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0
Note: you may need to restart the kernel to use updated packages.


# **mistral english GRPO**

In [11]:
# !rm -r /kaggle/working/ministral_base/__kaggle__temp__ministral_base_text

In [12]:
import json, os, glob
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
from transformers import AutoConfig, AutoTokenizer

REPO = "arefe-rn/mistral_v0_finetune_16bit"
OUT = "/kaggle/working/ministral_text"
os.makedirs(OUT, exist_ok=True)

src = snapshot_download(REPO)
cfg = AutoConfig.from_pretrained(src)
print("model_type:", cfg.model_type, "| text:", cfg.text_config.model_type)

# Look at actual key names first
first = sorted(glob.glob(f"{src}/*.safetensors"))[0]
print(list(load_file(first).keys())[:8])

def remap(k):
    if "vision_tower" in k or "multi_modal_projector" in k:
        return None
    for pre in ("language_model.model.", "model.language_model."):
        if k.startswith(pre):
            return "model." + k[len(pre):]
    if k.startswith("language_model.lm_head."):
        return "lm_head." + k[len("language_model.lm_head."):]
    return k

weight_map = {}
for f in sorted(glob.glob(f"{src}/*.safetensors")):
    sd = load_file(f)
    new = {remap(k): v.contiguous() for k, v in sd.items() if remap(k) is not None}
    name = os.path.basename(f)
    save_file(new, f"{OUT}/{name}", metadata={"format": "pt"})
    weight_map.update({k: name for k in new})
    del sd, new

json.dump({"metadata": {}, "weight_map": weight_map},
          open(f"{OUT}/model.safetensors.index.json", "w"))

# Keep the text config as-is (model_type stays "ministral3"), just fix the architecture
tc = cfg.text_config.to_dict()
tc["architectures"] = ["Ministral3ForCausalLM"]
tc["torch_dtype"] = "bfloat16"
json.dump(tc, open(f"{OUT}/config.json", "w"), indent=2)

AutoTokenizer.from_pretrained(src).save_pretrained(OUT)
print("done:", len(weight_map), "tensors")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}


model_type: mistral3 | text: ministral
['language_model.model.embed_tokens.weight', 'language_model.model.layers.0.input_layernorm.weight', 'language_model.model.layers.0.mlp.down_proj.weight', 'language_model.model.layers.0.mlp.gate_proj.weight', 'language_model.model.layers.0.mlp.up_proj.weight', 'language_model.model.layers.0.post_attention_layernorm.weight', 'language_model.model.layers.0.self_attn.k_proj.weight', 'language_model.model.layers.0.self_attn.o_proj.weight']


SafetensorError: Error while serializing: I/O error: No space left on device (os error 28)

In [ ]:
import torch
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained("/kaggle/working/ministral_text",
        torch_dtype=torch.bfloat16, device_map="auto")
print(type(m))

In [13]:
import gc, torch
try: del m
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

In [14]:
import json

p = "/kaggle/working/ministral_text/config.json"
c = json.load(open(p))
print("before:", c.get("model_type"), c.get("sliding_window"),
      c.get("max_position_embeddings"), (c.get("layer_types") or [None])[:3])

c["sliding_window"] = c.get("max_position_embeddings") or 262144
c["layer_types"] = ["full_attention"] * c["num_hidden_layers"]

json.dump(c, open(p, "w"), indent=2)
print("after:", c["sliding_window"], c["layer_types"][:3])

before: ministral 262144 262144 ['full_attention', 'full_attention', 'full_attention']
after: 262144 ['full_attention', 'full_attention', 'full_attention']


In [15]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --tasks gsm8k \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral

2026-10-13:17:41:46 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-13:17:41:57 INFO     [_cli.run:413] Selected Tasks: ['gsm8k']
2026-10-13:17:41:59 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-13:17:41:59 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/working/ministral_text', 'parallelize': True, 'dtype': 'float16'}
2026-10-13:17:42:06 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
2026-10-13:17:42:08 INFO     [models.huggingface:568] Model parallel wa

In [16]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --tasks gsm8k_cot \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral

2026-10-13:17:52:51 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-13:17:53:01 INFO     [_cli.run:413] Selected Tasks: ['gsm8k_cot']
2026-10-13:17:53:03 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-13:17:53:03 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/working/ministral_text', 'parallelize': True, 'dtype': 'float16'}
2026-10-13:17:53:09 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
2026-10-13:17:53:12 INFO     [models.huggingface:568] Model paralle

In [17]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --tasks gsm8k_cot_zeroshot \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral

2026-10-13:18:00:53 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-13:18:01:04 INFO     [_cli.run:413] Selected Tasks: ['gsm8k_cot_zeroshot']
2026-10-13:18:01:07 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-13:18:01:07 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/working/ministral_text', 'parallelize': True, 'dtype': 'float16'}
2026-10-13:18:01:15 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
2026-10-13:18:01:17 INFO     [models.huggingface:568] Mode

In [24]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --tasks gsm8k_cot_llama \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral

2026-10-14:17:16:23 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:17:16:32 INFO     [_cli.run:413] Selected Tasks: ['gsm8k_cot_llama']
2026-10-14:17:16:34 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-14:17:16:34 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/working/ministral_text', 'parallelize': True, 'dtype': 'float16'}
2026-10-14:17:16:39 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
2026-10-14:17:16:42 INFO     [models.huggingface:568] Model p

In [18]:
!accelerate launch --num_processes=1 -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --tasks persian_gsm8k \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/persian

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-13:18:10:01 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-13:18:10:11 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-13:18:10:11 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-13:18:10:11 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-13:18:10:11 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k']
2026-10-13:18:10:12 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setti

In [19]:
!accelerate launch --num_processes=1 -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --tasks persian_gsm8k_cot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/persian

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-13:18:22:17 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-13:18:22:28 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-13:18:22:28 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-13:18:22:28 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-13:18:22:28 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k_cot']
2026-10-13:18:22:29 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | S

In [20]:
!accelerate launch --num_processes=1 -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --tasks persian_gsm8k_cot_zeroshot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/persian

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-13:18:31:17 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-13:18:31:27 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-13:18:31:27 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-13:18:31:27 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-13:18:31:27 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k_cot_zeroshot']
2026-10-13:18:31:28 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to

## hendricksmath

In [17]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/working/ministral_text,parallelize=True,dtype=float16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --batch_size 2 \
    --device cuda:0 \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-19:12:55:34 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-19:12:55:45 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-19:12:55:47 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-19:12:55:47 WARNING  [evaluator:226] generation_kwargs: {'ma

# **mistral base** monde

In [3]:
pip install --upgrade git+https://github.com/huggingface/transformers.git

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-65s3iyzz
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-65s3iyzz
  Resolved https://github.com/huggingface/transformers.git to commit 6b75aeeab91d98bd3872b3f2382cfb4a8ce810d4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.21.0.dev0-py3-none-any.whl size=14236571 sha256=5e49a9d3741e892aaf3b70ad4b62734fd56730ce8927540e40a1aead06a662aa
  Stored in directory: /tmp/pip-ephem-wheel-cache-0xawil37/wheels/90/86/bb/8ad35b428289429952d8316d30f64abedae0ab2fcee0f4f46f
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
Note: you may need to resta

In [4]:
import json, os, glob
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
from transformers import AutoConfig, AutoTokenizer

REPO = "unsloth/Ministral-3-8B-Base-2512"
OUT = "/kaggle/temp/ministral_base_text"
os.makedirs(OUT, exist_ok=True)

src = snapshot_download(REPO)
cfg = AutoConfig.from_pretrained(src)
print("model_type:", cfg.model_type, "| text:", cfg.text_config.model_type)
print(list(load_file(sorted(glob.glob(f"{src}/*.safetensors"))[0]).keys())[:8])

def remap(k):
    if "vision_tower" in k or "multi_modal_projector" in k:
        return None
    for pre in ("language_model.model.", "model.language_model."):
        if k.startswith(pre):
            return "model." + k[len(pre):]
    if k.startswith("language_model.lm_head."):
        return "lm_head." + k[len("language_model.lm_head."):]
    return k

weight_map = {}
for f in sorted(glob.glob(f"{src}/*.safetensors")):
    sd = load_file(f)
    new = {remap(k): v.contiguous() for k, v in sd.items() if remap(k) is not None}
    name = os.path.basename(f)
    save_file(new, f"{OUT}/{name}", metadata={"format": "pt"})
    weight_map.update({k: name for k in new})
    del sd, new

json.dump({"metadata": {}, "weight_map": weight_map},
          open(f"{OUT}/model.safetensors.index.json", "w"))

tc = cfg.text_config.to_dict()
tc["architectures"] = ["Ministral3ForCausalLM"]
tc["torch_dtype"] = "bfloat16"
# same sliding-window workaround as before (full attention)
tc["sliding_window"] = tc.get("sliding_window") or tc.get("max_position_embeddings") or 262144
tc["layer_types"] = ["full_attention"] * tc["num_hidden_layers"]
json.dump(tc, open(f"{OUT}/config.json", "w"), indent=2)

# tokenizer with the regex fix
AutoTokenizer.from_pretrained(src, fix_mistral_regex=True).save_pretrained(OUT)
print("done:", len(weight_map), "tensors")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

model_type: mistral3 | text: ministral3
['language_model.model.embed_tokens.weight', 'language_model.model.layers.0.input_layernorm.weight', 'language_model.model.layers.0.mlp.down_proj.weight', 'language_model.model.layers.0.mlp.gate_proj.weight', 'language_model.model.layers.0.mlp.up_proj.weight', 'language_model.model.layers.0.post_attention_layernorm.weight', 'language_model.model.layers.0.self_attn.k_proj.weight', 'language_model.model.layers.0.self_attn.o_proj.weight']
done: 309 tensors


In [16]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=bfloat16 \
    --tasks gsm8k \
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_base

2026-10-14:07:10:45 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:07:10:55 INFO     [_cli.run:413] Selected Tasks: ['gsm8k']
2026-10-14:07:10:56 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-14:07:10:56 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/ministral_base_text', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-14:07:11:01 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Detected Mistral model with layer_types. Consider using AutoModel or Ministral classes instead to enable alternating attention compatibility.
2026-10-14:07:11:04 INFO     [models.huggingface:568] Model parallel was set to True, setting max memory per GPU to {

In [17]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=bfloat16 \
    --tasks gsm8k_cot \
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_base

2026-10-14:08:21:32 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:08:21:41 INFO     [_cli.run:413] Selected Tasks: ['gsm8k_cot']
2026-10-14:08:21:43 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-14:08:21:43 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/ministral_base_text', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-14:08:21:48 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Detected Mistral model with layer_types. Consider using AutoModel or Ministral classes instead to enable alternating attention compatibility.
2026-10-14:08:21:51 INFO     [models.huggingface:568] Model parallel was set to True, setting max memory per GPU 

In [19]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=bfloat16 \
    --tasks gsm8k_cot_zeroshot \
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_base

2026-10-14:10:36:03 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:10:36:13 INFO     [_cli.run:413] Selected Tasks: ['gsm8k_cot_zeroshot']
2026-10-14:10:36:14 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-14:10:36:14 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/ministral_base_text', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-14:10:36:21 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Detected Mistral model with layer_types. Consider using AutoModel or Ministral classes instead to enable alternating attention compatibility.
2026-10-14:10:36:23 INFO     [models.huggingface:568] Model parallel was set to True, setting max memory

In [23]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=bfloat16 \
    --tasks gsm8k_cot_llama \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_base

2026-10-14:15:49:48 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:15:49:57 INFO     [_cli.run:413] Selected Tasks: ['gsm8k_cot_llama']
2026-10-14:15:49:59 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-14:15:49:59 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/ministral_base_text', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-14:15:50:04 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Detected Mistral model with layer_types. Consider using AutoModel or Ministral classes instead to enable alternating attention compatibility.
2026-10-14:15:50:06 INFO     [models.huggingface:568] Model parallel was set to True, setting max memory pe

In [20]:
!accelerate launch --num_processes=1 -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=bfloat16 \
    --tasks persian_gsm8k \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 4 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_base

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-14:11:56:22 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:11:56:31 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-14:11:56:31 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-14:11:56:31 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-14:11:56:31 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k']
2026-10-14:11:56:33 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setti

In [21]:
!accelerate launch --num_processes=1 -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=bfloat16 \
    --tasks persian_gsm8k_cot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_base

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-14:12:59:54 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:13:00:04 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-14:13:00:04 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-14:13:00:04 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-14:13:00:04 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k_cot']
2026-10-14:13:00:05 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | S

In [22]:
!accelerate launch --num_processes=1 -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=bfloat16 \
    --tasks persian_gsm8k_cot_zeroshot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_base

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-14:14:00:38 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-14:14:00:48 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-14:14:00:48 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-14:14:00:48 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-14:14:00:48 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k_cot_zeroshot']
2026-10-14:14:00:49 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to

## hendricsMath

In [7]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=float16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 5 \
    --log_samples \
    --batch_size 2 \
    --device cuda:0 \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-21:11:21:31 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-21:11:21:42 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-21:11:21:44 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-21:11:21:44 WARNING  [evaluator:226] generation_kwargs: {'ma

In [8]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/ministral_base_text,parallelize=True,dtype=float16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --batch_size 2 \
    --device cuda:0 \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-21:11:43:07 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-21:11:43:19 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-21:11:43:21 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-21:11:43:21 WARNING  [evaluator:226] generation_kwargs: {'ma

# **mistral persian GRPO**

In [2]:
import json, os, glob, shutil
from huggingface_hub import snapshot_download
from safetensors.torch import load_file, save_file
from transformers import AutoConfig, AutoTokenizer

REPO = "arefe-rn/mistral_v0_finetune_16bit_persian"
OUT = "/kaggle/temp/mistral_v0_persian"
os.makedirs(OUT, exist_ok=True)

src = snapshot_download(REPO)
cfg = AutoConfig.from_pretrained(src)
print("model_type:", cfg.model_type, "| archs:", cfg.architectures)
print("sliding_window:", getattr(cfg, "sliding_window", None),
      "| tie:", getattr(cfg, "tie_word_embeddings", None))

files = sorted(glob.glob(f"{src}/*.safetensors"))
print(list(load_file(files[0]).keys())[:8])

def remap(k):
    # فقط اگر مدل چندوجهی/wrapper بود لازم می‌شود
    if "vision_tower" in k or "multi_modal_projector" in k:
        return None
    for pre in ("language_model.model.", "model.language_model."):
        if k.startswith(pre):
            return "model." + k[len(pre):]
    if k.startswith("language_model.lm_head."):
        return "lm_head." + k[len("language_model.lm_head."):]
    return k

# وزن‌ها: اگر کلیدها از قبل استاندارد باشند، همان‌طور کپی می‌شوند
weight_map = {}
for f in files:
    sd = load_file(f)
    new = {remap(k): v.contiguous() for k, v in sd.items() if remap(k) is not None}
    name = os.path.basename(f)
    save_file(new, f"{OUT}/{name}", metadata={"format": "pt"})
    weight_map.update({k: name for k in new})
    del sd, new

json.dump({"metadata": {}, "weight_map": weight_map},
          open(f"{OUT}/model.safetensors.index.json", "w"))

# کانفیگ: بدون دستکاری، مگر اینکه wrapper باشد
c = cfg.to_dict()
if "text_config" in c:
    c = cfg.text_config.to_dict()
    c["architectures"] = ["MistralForCausalLM"]
c["torch_dtype"] = "bfloat16"
json.dump(c, open(f"{OUT}/config.json", "w"), indent=2)

# generation_config در صورت وجود
gc = f"{src}/generation_config.json"
if os.path.exists(gc):
    shutil.copy(gc, OUT)

# tokenizer (fix_mistral_regex فقط در نسخه‌های جدید transformers وجود دارد)
try:
    tok = AutoTokenizer.from_pretrained(src, fix_mistral_regex=True)
except TypeError:
    tok = AutoTokenizer.from_pretrained(src)
tok.save_pretrained(OUT)
print("done:", len(weight_map), "tensors")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}


model_type: mistral3 | archs: ['Mistral3ForConditionalGeneration']
sliding_window: None | tie: True
['language_model.model.embed_tokens.weight', 'language_model.model.layers.0.input_layernorm.weight', 'language_model.model.layers.0.mlp.down_proj.weight', 'language_model.model.layers.0.mlp.gate_proj.weight', 'language_model.model.layers.0.mlp.up_proj.weight', 'language_model.model.layers.0.post_attention_layernorm.weight', 'language_model.model.layers.0.self_attn.k_proj.weight', 'language_model.model.layers.0.self_attn.o_proj.weight']


[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}


done: 309 tensors


In [3]:
c = cfg.to_dict()
if "text_config" in c:
    c = cfg.text_config.to_dict()

c["architectures"] = ["Ministral3ForCausalLM"]
c["torch_dtype"] = "bfloat16"
# workaround: full attention
c["sliding_window"] = c.get("sliding_window") or c.get("max_position_embeddings") or 262144
c["layer_types"] = ["full_attention"] * c["num_hidden_layers"]
json.dump(c, open(f"{OUT}/config.json", "w"), indent=2)

In [8]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/mistral_v0_persian,parallelize=True,dtype=bfloat16 \
    --tasks gsm8k gsm8k_cot gsm8k_cot_zeroshot\
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_grpo_persian

2026-10-18:17:53:47 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-18:17:53:58 INFO     [_cli.run:413] Selected Tasks: ['gsm8k', 'gsm8k_cot', 'gsm8k_cot_zeroshot']
2026-10-18:17:54:00 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-18:17:54:00 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/mistral_v0_persian', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-18:17:54:07 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
2026-10-18:17:54:09 INFO     [mode

In [6]:
!python -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/mistral_v0_persian,parallelize=True,dtype=bfloat16 \
    --tasks  gsm8k_cot_llama\
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_grpo_persian

2026-10-19:07:37:20 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-19:07:37:31 INFO     [_cli.run:413] Selected Tasks: ['gsm8k_cot_llama']
2026-10-19:07:37:33 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-19:07:37:33 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': '/kaggle/temp/mistral_v0_persian', 'parallelize': True, 'dtype': 'bfloat16'}
2026-10-19:07:37:39 INFO     [models.huggingface:309] Using `accelerate launch` or `parallelize=True`, device 'cuda:0' will be overridden when placing model.
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='yarn': {'llama_4_scaling_beta'}
2026-10-19:07:37:42 INFO     [models.huggingface:568] Model

In [9]:
!accelerate launch --num_processes=1 -m lm_eval \
    --model hf \
    --model_args pretrained=/kaggle/temp/mistral_v0_persian,parallelize=True,dtype=bfloat16 \
    --tasks persian_gsm8k persian_gsm8k_cot persian_gsm8k_cot_zeroshot \
    --include_path /kaggle/input/datasets/amirint/pgsm8k \
    --device cuda:0 \
    --batch_size 2 \
    --limit 100 \
    --log_samples \
    --output_path /kaggle/working/ministral_grpo_persian

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-18:21:04:38 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-18:21:04:47 WARNING  [tasks._index:70] Task 'chain_of_thought' from None overrides existing task from None
2026-10-18:21:04:47 WARNING  [tasks._index:70] Task 'math_word_problems' from None overrides existing task from None
2026-10-18:21:04:47 INFO     [_cli.run:412] Including path: /kaggle/input/datasets/amirint/pgsm8k
2026-10-18:21:04:47 INFO     [_cli.run:413] Selected Tasks: ['persian_gsm8k', 'persian_gsm8k_cot', 'persian_gsm8k_cot_zeroshot']
2026-10-18:21:04:49 INFO     [evaluator:214] Setting ra

## henricksmath

In [5]:
%%writefile /kaggle/working/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py
from typing import Dict, List

import datasets


def process_docs(dataset: datasets.Dataset) -> datasets.Dataset:
    def _process_doc(doc: dict) -> dict:
        out_doc = {
            "problem": doc["problem"],
            "solution": doc["solution"],
            "answer": remove_boxed(last_boxed_only_string(doc["solution"])),
        }
        return out_doc

    return dataset.map(_process_doc)


def process_results(doc: dict, results: List[str]) -> Dict[str, int]:
    retval = 0
    response = results[0]

    # Try to extract \boxed{} from the response (matches aime task behavior)
    boxed = last_boxed_only_string(response)
    if boxed is not None:
        try:
            answer = remove_boxed(boxed)
        except AssertionError:
            answer = None
    else:
        answer = None

    # Fall back to $...$ extraction
    if answer is None:
        indices = [pos for pos, char in enumerate(response) if char == "$"]
        if len(indices) <= 1:
            answer = response
        else:
            answer = response[indices[0] + 1 : indices[-1]]

    if is_equiv(answer, remove_boxed(last_boxed_only_string(doc["solution"]))):
        retval = 1

    results = {
        "exact_match": retval,
    }
    return results


# string normalization from https://github.com/EleutherAI/lm-evaluation-harness/blob/master/lm_eval/tasks/hendrycks_math.py
def is_equiv(str1, str2, verbose=False):
    if str1 is None and str2 is None:
        print("WARNING: Both None")
        return True
    if str1 is None or str2 is None:
        return False

    try:
        ss1 = strip_string(str1)
        ss2 = strip_string(str2)
        if verbose:
            print(ss1, ss2)
        return ss1 == ss2
    except Exception:
        return str1 == str2


def remove_boxed(s):
    if "\\boxed " in s:
        left = "\\boxed "
        assert s[: len(left)] == left
        return s[len(left) :]

    left = "\\boxed{"

    assert s[: len(left)] == left
    assert s[-1] == "}"

    return s[len(left) : -1]


def last_boxed_only_string(string):
    idx = string.rfind("\\boxed")
    if "\\boxed " in string:
        return "\\boxed " + string.split("\\boxed ")[-1].split("$")[0]
    if idx < 0:
        idx = string.rfind("\\fbox")
        if idx < 0:
            return None

    i = idx
    right_brace_idx = None
    num_left_braces_open = 0
    while i < len(string):
        if string[i] == "{":
            num_left_braces_open += 1
        if string[i] == "}":
            num_left_braces_open -= 1
            if num_left_braces_open == 0:
                right_brace_idx = i
                break
        i += 1

    if right_brace_idx is None:
        retval = None
    else:
        retval = string[idx : right_brace_idx + 1]

    return retval


def fix_fracs(string):
    substrs = string.split("\\frac")
    new_str = substrs[0]
    if len(substrs) > 1:
        substrs = substrs[1:]
        for substr in substrs:
            new_str += "\\frac"
            if substr[0] == "{":
                new_str += substr
            else:
                try:
                    assert len(substr) >= 2
                except AssertionError:
                    return string
                a = substr[0]
                b = substr[1]
                if b != "{":
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}{" + b + "}" + post_substr
                    else:
                        new_str += "{" + a + "}{" + b + "}"
                else:
                    if len(substr) > 2:
                        post_substr = substr[2:]
                        new_str += "{" + a + "}" + b + post_substr
                    else:
                        new_str += "{" + a + "}" + b
    string = new_str
    return string


def fix_a_slash_b(string):
    if len(string.split("/")) != 2:
        return string
    a = string.split("/")[0]
    b = string.split("/")[1]
    try:
        a = int(a)
        b = int(b)
        assert string == "{}/{}".format(a, b)
        new_string = "\\frac{" + str(a) + "}{" + str(b) + "}"
        return new_string
    except AssertionError:
        return string


def remove_right_units(string):
    # "\\text{ " only ever occurs (at least in the val set) when describing units
    if "\\text{ " in string:
        splits = string.split("\\text{ ")
        assert len(splits) == 2
        return splits[0]
    else:
        return string


def fix_sqrt(string):
    if "\\sqrt" not in string:
        return string
    splits = string.split("\\sqrt")
    new_string = splits[0]
    for split in splits[1:]:
        if split[0] != "{":
            a = split[0]
            new_substr = "\\sqrt{" + a + "}" + split[1:]
        else:
            new_substr = "\\sqrt" + split
        new_string += new_substr
    return new_string


def strip_string(string):
    # linebreaks
    string = string.replace("\n", "")

    # remove inverse spaces
    string = string.replace("\\!", "")

    # replace \\ with \
    string = string.replace("\\\\", "\\")

    # replace tfrac and dfrac with frac
    string = string.replace("tfrac", "frac")
    string = string.replace("dfrac", "frac")

    # remove \left and \right
    string = string.replace("\\left", "")
    string = string.replace("\\right", "")

    # Remove circ (degrees)
    string = string.replace("^{\\circ}", "")
    string = string.replace("^\\circ", "")

    # remove dollar signs
    string = string.replace("\\$", "")

    # remove units (on the right)
    string = remove_right_units(string)

    # remove percentage
    string = string.replace("\\%", "")
    string = string.replace("\%", "")  # noqa: W605

    # " 0." equivalent to " ." and "{0." equivalent to "{." Alternatively, add "0" if "." is the start of the string
    string = string.replace(" .", " 0.")
    string = string.replace("{.", "{0.")
    # if empty, return empty string
    if len(string) == 0:
        return string
    if string[0] == ".":
        string = "0" + string

    # to consider: get rid of e.g. "k = " or "q = " at beginning
    if len(string.split("=")) == 2:
        if len(string.split("=")[0]) <= 2:
            string = string.split("=")[1]

    # fix sqrt3 --> sqrt{3}
    string = fix_sqrt(string)

    # remove spaces
    string = string.replace(" ", "")

    # \frac1b or \frac12 --> \frac{1}{b} and \frac{1}{2}, etc. Even works with \frac1{72} (but not \frac{72}1). Also does a/b --> \\frac{a}{b}
    string = fix_fracs(string)

    # manually change 0.5 --> \frac{1}{2}
    if string == "0.5":
        string = "\\frac{1}{2}"

    # NOTE: X/Y changed to \frac{X}{Y} in dataset, but in simple cases fix in case the model output is X/Y
    string = fix_a_slash_b(string)

    return string

Overwriting /kaggle/working/lm-evaluation-harness/lm_eval/tasks/hendrycks_math/utils.py


In [11]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/mistral_v0_persian,parallelize=True,dtype=bfloat16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 5 \
    --log_samples \
    --batch_size 2 \
    --device cuda:0 \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-19:09:57:33 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-19:09:57:44 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-19:09:57:46 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-19:09:57:46 WARNING  [evaluator:226] generation_kwargs: {'ma

In [7]:
!accelerate launch --num_processes=1 -m lm_eval --model hf \
    --tasks hendrycks_math_algebra hendrycks_math_counting_and_prob hendrycks_math_geometry hendrycks_math_intermediate_algebra hendrycks_math_num_theory hendrycks_math_prealgebra hendrycks_math_precalc \
    --model_args pretrained=/kaggle/temp/mistral_v0_persian,parallelize=True,dtype=float16 \
    --gen_kwargs max_new_tokens=1024 \
    --limit 100 \
    --log_samples \
    --batch_size 2 \
    --device cuda:0 \
    --output_path /kaggle/working/results/hendricks \
    --system_instruction "Solve the following mathematics problem. Show your reasoning step by step. Use standard mathematical notation for fractions, radicals, equations, and other expressions. Put your final answer in \boxed{...}."

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
2026-10-20:13:56:37 WARNING  [config.evaluate_config:344] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-10-20:13:56:49 INFO     [_cli.run:413] Selected Tasks: ['hendrycks_math_algebra', 'hendrycks_math_counting_and_prob', 'hendrycks_math_geometry', 'hendrycks_math_intermediate_algebra', 'hendrycks_math_num_theory', 'hendrycks_math_prealgebra', 'hendrycks_math_precalc']
2026-10-20:13:56:50 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-20:13:56:50 WARNING  [evaluator:226] generation_kwargs: {'ma